# S-Pol Radar Reflectivity from the PRECIP Field Campaign

```{image} ../../thumbnails/gdex_logo.png
:alt: GDEX Cookbook logo
:width: 200px
```

---

## Overview

During the 2022 Prediction of Rainfall Extremes Campaign In the Pacific (PRECIP),
NCAR's S-Pol radar scanned precipitating systems over the western Pacific. GDEX
hosts the radar moments as the ARCO dataset [`d694517`](https://gdex.ucar.edu/datasets/d694517/),
converted to analysis-ready **Zarr** stores following the **CfRadial2** convention
and served over OSDF.

S-Pol scans in two geometries: vertical **Range–Height Indicator (RHI)** slices and
horizontal **Plan Position Indicator (PPI)** surveillance sweeps. We open one of
each and plot reflectivity.

1. Open S-Pol radar Zarr stores as a hierarchical `DataTree`
2. Extract a single sweep
3. Plot an RHI (vertical) reflectivity cross-section
4. Georeference and map a PPI (horizontal) sweep

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [Introduction to ARCO/Zarr on GDEX](../services/zarr_intro.ipynb) | Helpful | Opening Zarr stores on GDEX |
| [xarray](https://docs.xarray.dev) | Necessary | `DataTree` for hierarchical sweeps |
| [xradar](https://docs.openradarscience.org/projects/xradar/) | Necessary | Radar data structures & georeferencing |
| [cmweather](https://cmweather.readthedocs.io/) | Necessary | Weather colormaps |
| [Cartopy](https://scitools.org.uk/cartopy) | Necessary | Mapping |

- **Time to learn**: 30 minutes
- **System requirements**: `zarr ≥ 3.1.6` and a recent `xarray` with `DataTree` support

---

## Imports

In [1]:
import numpy as np
import xarray as xr
import xradar as xd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmweather  

## Locating the data

S-Pol stores one Zarr per scan. We use two scans from 25 May 2022 — an RHI
(vertical) scan and a PPI (horizontal surveillance) scan — streamed from GDEX over
the OSDF director.

In [4]:
base = "https://osdf-director.osg-htc.org/ncar-gdex/d694517/v2.0_zarr"
#base = 'osdf:///ncar-gdex/d694517/v2.0_zarr'
rhi_url = f"{base}/cfrad.20220525_015443.665_to_20220525_015709.985_SPOL_PrecipRhi2_RHI.zarr"
ppi_url = f"{base}/cfrad.20220525_030910.927_to_20220525_031137.777_SPOL_PrecipSur2_SUR.zarr"

### Opening the data

CfRadial2 organizes each scan's sweeps as a hierarchy, so we open the stores with
`xr.open_datatree` — every sweep (`sweep_0`, `sweep_1`, …) is a node in the tree.

In [5]:
dt_rhi = xr.open_datatree(rhi_url, engine="zarr", decode_timedelta=False)
dt_ppi = xr.open_datatree(ppi_url, engine="zarr", decode_timedelta=False)
dt_rhi

<xarray.DataTree>
Group: /
│   Dimensions:              (sweep: 11, frequency: 1)
│   Coordinates:
│     * frequency            (frequency) float32 4B 2.81e+09
│   Dimensions without coordinates: sweep
│   Data variables: (12/13)
│       altitude             float64 8B ...
│       altitude_agl         float64 8B ...
│       instrument_type      |S32 32B ...
│       latitude             float64 8B ...
│       longitude            float64 8B ...
│       platform_type        |S32 32B ...
│       ...                   ...
│       status_str           |S10082 10kB ...
│       sweep_fixed_angle    (sweep) float32 44B ...
│       sweep_group_name     (sweep) object 88B ...
│       time_coverage_end    |S32 32B ...
│       time_coverage_start  |S32 32B ...
│       volume_number        float64 8B ...
│   Attributes: (12/14)
│       Conventions:         CF-1.7
│       version:             CF-Radial-1.4
│       title:               
│       institution:         
│       references:          
│       source:              
│       ...                  ...
│       instrument_name:     SPOL
│       site_name:           Nanliao
│       scan_name:           PrecipRhi2
│       scan_id:             0
│       platform_is_mobile:  false
│       ray_times_increase:  true
├── Group: /georeferencing_correction
├── Group: /radar_calibration
│       Dimensions:                   ()
│       Data variables: (12/55)
│           antenna_gain_h            float32 4B ...
│           antenna_gain_v            float32 4B ...
│           base_1km_hc               float32 4B ...
│           base_1km_hx               float32 4B ...
│           base_1km_vc               float32 4B ...
│           base_1km_vx               float32 4B ...
│           ...                        ...
│           two_way_radome_loss_v     float32 4B ...
│           two_way_waveguide_loss_h  float32 4B ...
│           two_way_waveguide_loss_v  float32 4B ...
│           xmit_power_h              float32 4B ...
│           xmit_power_v              float32 4B ...
│           zdr_correction            float32 4B ...
├── Group: /radar_parameters
│       Dimensions:                   ()
│       Data variables:
│           radar_antenna_gain_h      float32 4B ...
│           radar_antenna_gain_v      float32 4B ...
│           radar_beam_width_h        float32 4B ...
│           radar_beam_width_v        float32 4B ...
│           radar_receiver_bandwidth  float32 4B ...
├── Group: /sweep_0
│       Dimensions:                    (azimuth: 252, range: 1999)
│       Coordinates:
│         * azimuth                    (azimuth) float32 1kB 90.03 90.03 ... 90.04 90.04
│           elevation                  (azimuth) float32 1kB ...
│           time                       (azimuth) datetime64[ns] 2kB ...
│         * range                      (range) float32 8kB 75.0 225.0 ... 2.998e+05
│           altitude                   float64 8B ...
│           latitude                   float64 8B ...
│           longitude                  float64 8B ...
│       Data variables: (12/46)
│           antenna_transition         (azimuth) float32 1kB ...
│           CMD_FLAG                   (azimuth, range) float32 2MB ...
│           DBMHC                      (azimuth, range) float32 2MB ...
│           DBMHC_F                    (azimuth, range) float32 2MB ...
│           DBMVC                      (azimuth, range) float32 2MB ...
│           DBMVC_F                    (azimuth, range) float32 2MB ...
│           ...                         ...
│           unambiguous_range          (azimuth) float32 1kB ...
│           VEL                        (azimuth, range) float32 2MB ...
│           VEL_F                      (azimuth, range) float32 2MB ...
│           WIDTH                      (azimuth, range) float32 2MB ...
│           WIDTH_F                    (azimuth, range) float32 2MB ...
│           ZDR_F                      (azimuth, range) float32 2MB ...
├── Group: /sweep_1
│       Dimensions:      

## Visualizing the RHI (vertical) scan

We extract the first sweep, compute each gate's altitude from its range and
elevation angle (accounting for Earth's curvature), and plot reflectivity as a
range–altitude cross-section.

In [ ]:
ds_rhi = dt_rhi["sweep_0"].to_dataset()


def rhi_altitude(r, elevation_deg):
    """Beam altitude (km) from range (m) and elevation angle (deg), with Earth curvature."""
    r_earth = 6_371_000.0
    e = np.deg2rad(elevation_deg)
    return (np.sqrt(r**2 + r_earth**2 + 2 * r * r_earth * np.sin(e)) - r_earth) / 1000.0


ds_rhi["rhi_alt"] = rhi_altitude(ds_rhi["range"], ds_rhi["elevation"]).compute()
ds_rhi = ds_rhi.set_coords("rhi_alt")
ds_rhi["range"] = ds_rhi["range"] / 1000.0
ds_rhi["rhi_alt"].attrs = {"long_name": "RHI altitude", "units": "km"}
ds_rhi["range"].attrs["units"] = "km"

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ds_rhi["DBZ"].plot.pcolormesh(
    x="range",
    y="rhi_alt",
    cmap="ChaseSpectral",
    levels=np.arange(-10, 55, 1),
    ax=ax,
    cbar_kwargs={"label": "Reflectivity (dBZ)"},
)
ax.set_xlabel("Range (km)")
ax.set_ylabel("Altitude (km)")
ax.set_xlim(0, 150)
ax.set_ylim(0, 15)
ax.set_title(
    f"S-Pol RHI reflectivity — 2022-05-25 "
    f"(lon {ds_rhi.longitude.values:.2f}, lat {ds_rhi.latitude.values:.2f})"
)
plt.show()

## Visualizing the PPI (horizontal) scan

For the surveillance scan we use `xradar` to georeference the polar coordinates into
Cartesian `x`/`y`/`z`, then map reflectivity onto a Cartopy projection for
geographic context.

In [ ]:
dt_ppi = dt_ppi.xradar.georeference()
dt_ppi = xd.georeference.get_x_y_z_tree(dt_ppi)
ds_ppi = dt_ppi["sweep_0"].to_dataset()
da_dbz = ds_ppi["DBZ"]

In [ ]:
radar_crs = ccrs.Projection(xd.georeference.get_crs(ds_ppi))

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
da_dbz.plot(
    x="x",
    y="y",
    transform=radar_crs,
    cmap="ChaseSpectral",
    levels=np.arange(-10, 55, 1),
    cbar_kwargs={"label": "Reflectivity (dBZ)", "shrink": 0.75, "pad": 0.075},
)
ax.coastlines()
ax.gridlines(draw_labels=True)
ax.set_title(f"S-Pol PPI reflectivity — sweep 0 (elevation {ds_ppi.elevation.values[0]:.2f}°)")
plt.show()

---

## Summary

We opened NCAR S-Pol radar scans from the PRECIP campaign as CfRadial2 Zarr
`DataTree`s over OSDF, extracted single sweeps, and plotted reflectivity as a
vertical RHI cross-section and a georeferenced horizontal PPI map.

### What's next?

Continue to [Data Fusion](data_fusion.ipynb) to combine multiple GDEX datasets in a
single analysis.

## Resources and references

- [GDEX S-Pol PRECIP dataset (`d694517`)](https://gdex.ucar.edu/datasets/d694517/)
- [Original GDEX EOL radar example](https://ncar.github.io/gdex-examples/eol-radar-precip/)
- [xradar documentation](https://docs.openradarscience.org/projects/xradar/)
- [cmweather](https://cmweather.readthedocs.io/)
- [CfRadial2 / FM-301 background (EarthMover)](https://earthmover.io/blog/from-files-to-datasets-fm-301-and-the-future-of-radar-interoperability)